In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score

import lightgbm as lgb
import joblib

from model import DemandModel

In [2]:
df = pd.read_csv("dataset/train.csv", index_col='Index')
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
# splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# tv_idx, test_idx = next(
#     splitter.split(df, groups=df["geohash"])
# )

# tv_df = df.iloc[tv_idx].reset_index(drop=True)
# test_df = df.iloc[test_idx].reset_index(drop=True)

In [4]:
from sklearn.model_selection import train_test_split

tv_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [5]:
tv_df['geohash'].value_counts().to_dict()

{'qp09bh': 94,
 'qp092q': 92,
 'qp098p': 92,
 'qp03qj': 92,
 'qp0985': 91,
 'qp0921': 91,
 'qp03we': 91,
 'qp03xq': 91,
 'qp03xt': 90,
 'qp0926': 89,
 'qp09de': 89,
 'qp03y8': 89,
 'qp09d3': 89,
 'qp09uv': 89,
 'qp09se': 89,
 'qp097k': 88,
 'qp09d2': 88,
 'qp03zt': 88,
 'qp03yg': 88,
 'qp03yc': 88,
 'qp03r3': 88,
 'qp03xj': 88,
 'qp03yy': 88,
 'qp03qb': 87,
 'qp09fq': 87,
 'qp03wy': 87,
 'qp03rv': 87,
 'qp03xv': 87,
 'qp097v': 87,
 'qp03r6': 87,
 'qp09v0': 87,
 'qp03qc': 87,
 'qp03xp': 87,
 'qp09df': 87,
 'qp03qg': 87,
 'qp09b2': 87,
 'qp03xf': 87,
 'qp03wv': 87,
 'qp094q': 87,
 'qp03qh': 87,
 'qp03yd': 87,
 'qp0987': 86,
 'qp09d9': 86,
 'qp03ry': 86,
 'qp09hj': 86,
 'qp098h': 86,
 'qp03rz': 86,
 'qp09gk': 86,
 'qp03xh': 86,
 'qp097s': 86,
 'qp03r1': 86,
 'qp03xx': 86,
 'qp092s': 86,
 'qp092m': 86,
 'qp09b7': 86,
 'qp03yb': 86,
 'qp092h': 86,
 'qp03w8': 86,
 'qp03zb': 85,
 'qp03xn': 85,
 'qp03w0': 85,
 'qp09db': 85,
 'qp095v': 85,
 'qp03rm': 85,
 'qp03ws': 85,
 'qp03xs': 85,
 'qp09t0':

In [6]:
test_df['geohash'].value_counts().to_dict()

{'qp03q8': 29,
 'qp0dj0': 29,
 'qp03r9': 29,
 'qp09dj': 28,
 'qp098d': 28,
 'qp03wc': 28,
 'qp03qe': 28,
 'qp03qy': 28,
 'qp0d4b': 27,
 'qp09fj': 27,
 'qp09k3': 27,
 'qp09b5': 27,
 'qp03z8': 27,
 'qp096q': 27,
 'qp03zv': 27,
 'qp097e': 27,
 'qp09sd': 27,
 'qp03zz': 27,
 'qp0925': 27,
 'qp03x7': 27,
 'qp092r': 26,
 'qp03qd': 26,
 'qp0998': 26,
 'qp098e': 26,
 'qp03w9': 26,
 'qp03nx': 26,
 'qp03x3': 26,
 'qp096t': 26,
 'qp092d': 26,
 'qp097g': 26,
 'qp0920': 26,
 'qp09b4': 26,
 'qp06pb': 26,
 'qp099e': 26,
 'qp0dht': 26,
 'qp03ru': 26,
 'qp096m': 26,
 'qp09b0': 26,
 'qp09vm': 26,
 'qp09fu': 26,
 'qp03wu': 26,
 'qp03mv': 26,
 'qp09kv': 26,
 'qp09yh': 26,
 'qp03wz': 25,
 'qp09d7': 25,
 'qp03nr': 25,
 'qp03x5': 25,
 'qp0d0j': 25,
 'qp02zt': 25,
 'qp09vk': 25,
 'qp098r': 25,
 'qp03xd': 25,
 'qp0988': 25,
 'qp094r': 25,
 'qp03xm': 25,
 'qp09bj': 25,
 'qp09bm': 25,
 'qp09dk': 25,
 'qp03rr': 25,
 'qp096x': 25,
 'qp03qv': 25,
 'qp03ww': 25,
 'qp098k': 25,
 'qp09ek': 25,
 'qp097y': 25,
 'qp092k':

In [7]:
print(len(tv_df), len(test_df))

61839 15460


In [8]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.02,

    "num_leaves": 128,
    "max_depth": 8,
    "min_data_in_leaf": 30,

    "feature_fraction": 0.6,
    "bagging_fraction": 0.6,
    "bagging_freq": 5,

    "lambda_l1": 2.0,
    "lambda_l2": 8.0,

    "min_gain_to_split": 0.01,

    "verbosity": -1
}

In [9]:
from sklearn.model_selection import GroupKFold, KFold

x = tv_df.copy()
y = tv_df["demand"]

# gkf = GroupKFold(n_splits=4)
kf = KFold(n_splits=4,
           shuffle=True,
           random_state=42)
oof = np.zeros(len(tv_df))

for fold, (tr, va) in enumerate(kf.split(x, y)):

    train_fold = tv_df.iloc[tr]
    val_fold = tv_df.iloc[va]

    model = DemandModel(params)
    model.fit(train_fold, val_fold)

    preds = model.predict(val_fold)
    oof[va] = preds

    print(f"Fold {fold + 1} R2:", r2_score(val_fold["demand"], preds))

print("OOF R2:", r2_score(y, oof))

Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.0761822
[100]	valid_0's rmse: 0.0543715
[150]	valid_0's rmse: 0.0487309
[200]	valid_0's rmse: 0.046394
[250]	valid_0's rmse: 0.0455797
[300]	valid_0's rmse: 0.0451738
[350]	valid_0's rmse: 0.0450273
[400]	valid_0's rmse: 0.044893
[450]	valid_0's rmse: 0.044825
[500]	valid_0's rmse: 0.0447646
[550]	valid_0's rmse: 0.0447498
[600]	valid_0's rmse: 0.0447022
[650]	valid_0's rmse: 0.0446757
[700]	valid_0's rmse: 0.0446564
[750]	valid_0's rmse: 0.0446212
[800]	valid_0's rmse: 0.0446051
[850]	valid_0's rmse: 0.0446022
[900]	valid_0's rmse: 0.0446022
Early stopping, best iteration is:
[818]	valid_0's rmse: 0.0446022
Fold 1 R2: 0.9093653639681728
Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.0723455
[100]	valid_0's rmse: 0.0515223
[150]	valid_0's rmse: 0.0463399
[200]	valid_0's rmse: 0.0442328
[250]	valid_0's rmse: 0.0435874
[300]	valid_0's rmse: 0.0432573
[350]	valid_0's 

In [10]:
test_y = test_df["demand"]
test_x = test_df.drop(columns=["demand"])
preds = model.predict(test_x)

print("Test R2:", r2_score(test_y, preds))

Test R2: 0.9055843820597157


In [11]:
joblib.dump(model.model, "model.pkl")
joblib.dump(model.pipeline, "pipeline.pkl")

['pipeline.pkl']

### -----TESTING ROUTINE----- ###

In [12]:
submission_test_df = pd.read_csv("dataset/test.csv")
predictions = model.predict(submission_test_df)

submission = pd.DataFrame({
    "Index": submission_test_df.index,
    "demand": predictions
})

submission.to_csv("dataset/submission.csv", index=False)